In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LeakyReLU


REGRESSION

In [10]:
df=pd.read_csv("0019/DEEP LEARNING/concrete_data.csv")
df.head()

,cement,blast_furnace_slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,age,concrete_compressive_strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [12]:
X = df.iloc[:, :-1].values   
y = df.iloc[:, -1].values 


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [13]:
def build_regression_model(hidden_units=16, activation='relu'):
    model = Sequential()
    
 
    model.add(Dense(hidden_units, input_dim=X_train.shape[1], activation=activation))
    model.add(Dense(hidden_units, activation=activation))
    model.add(Dense(hidden_units, activation=activation))
   
    model.add(Dense(1, activation='linear'))
    
    model.compile(optimizer='adam', loss='mse')
    return model


In [17]:
activations = ['relu', 'tanh', 'sigmoid'] 

results = []

for act in activations:
    print(f"\n--- Activation: {act} ---")
    
    model = build_regression_model(hidden_units=16, activation=act)
    model.fit(X_train, y_train, epochs=200, batch_size=16, verbose=0)
    
    # Predictions
    y_pred = model.predict(X_test, verbose=0)
    
    # Metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    params = model.count_params()
    
    print(f"Parameters: {params}, MSE: {mse:.3f}, RMSE: {rmse:.3f}, MAE: {mae:.3f}, R²: {r2:.3f}")
    
    results.append((act, params, mse, rmse, mae, r2))



--- Activation: relu ---


/home/student/0019/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Parameters: 705, MSE: 40.973, RMSE: 6.401, MAE: 4.972, R²: 0.841

--- Activation: tanh ---


/home/student/0019/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Parameters: 705, MSE: 31.881, RMSE: 5.646, MAE: 4.093, R²: 0.876

--- Activation: sigmoid ---


/home/student/0019/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Parameters: 705, MSE: 80.266, RMSE: 8.959, MAE: 6.828, R²: 0.689


In [19]:
import pandas as pd

df_results = pd.DataFrame(results, columns=['Activation', 'Params', 'MSE', 'RMSE', 'MAE', 'R2'])

df_results


,Activation,Params,MSE,RMSE,MAE,R2
0,relu,705,40.972590,6.400984,4.972055,0.840992
1,tanh,705,31.880939,5.646321,4.093265,0.876276
2,sigmoid,705,80.265607,8.959107,6.828331,0.688503


classification

In [28]:
df=pd.read_csv("0019/DEEP LEARNING/glass.csv")
from sklearn.preprocessing import  LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [29]:

X = df.drop('Type', axis=1).values


y = df['Type'].values


le = LabelEncoder()
y_encoded = le.fit_transform(y)


y_onehot = tf.keras.utils.to_categorical(y_encoded)


X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [30]:
def build_classification_model(hidden_units=16, activation='relu'):
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=X_train.shape[1], activation=activation))
    model.add(Dense(hidden_units, activation=activation))
    model.add(Dense(hidden_units, activation=activation))
    

    n_classes = y_onehot.shape[1]
    model.add(Dense(n_classes, activation='softmax'))
    
    model.compile(optimizer='adam', loss='categorical_crossentropy')
    return model


In [31]:
activations = ['relu', 'tanh', 'sigmoid']


In [40]:
results = []

for act in activations:
    print(f"\n--- Activation: {act} ---")
    
    model = build_classification_model(hidden_units=32, activation=act)
    model.fit(X_train, y_train, epochs=200, batch_size=16, verbose=0)
    
    # Predictions
    y_pred_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = np.argmax(y_test, axis=1)
    
    # Metrics
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    params = model.count_params()
    
    print(f"Parameters: {params}, Accuracy: {acc:.3f}, Precision: {precision:.3f}, Recall: {recall:.3f}, F1-score: {f1:.3f}")
    
    results.append((act, params, acc, precision, recall, f1))



--- Activation: relu ---


/home/student/0019/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Parameters: 2630, Accuracy: 0.744, Precision: 0.788, Recall: 0.712, F1-score: 0.725

--- Activation: tanh ---


/home/student/0019/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Parameters: 2630, Accuracy: 0.791, Precision: 0.792, Recall: 0.783, F1-score: 0.785

--- Activation: sigmoid ---


/home/student/0019/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Parameters: 2630, Accuracy: 0.674, Precision: 0.658, Recall: 0.616, F1-score: 0.626


In [41]:
df_results = pd.DataFrame(results, columns=['Activation', 'Params', 'Accuracy', 'Precision', 'Recall', 'F1'])

df_results


,Activation,Params,Accuracy,Precision,Recall,F1
0,relu,2630,0.744186,0.788408,0.712482,0.724900
1,tanh,2630,0.790698,0.792448,0.783189,0.785290
2,sigmoid,2630,0.674419,0.657639,0.615981,0.625788


Tanh is the best hidden activation for this Glass dataset  because of High Accuracy (79%) , Balanced Precision

ReLU is decent in which Accuracy slightly lower than Tanh and Precision higher than Recall → overconfident on some classes

Sigmoid is unsuitable for this 3-hidden-layer NN [ likely underfitting due to vanishing gradients ]

Increasing hidden neurons (params = 2630) improved overall performance

Previously with 806 params, Accuracy was ~74–76%

Now with 2630 params, Tanh Accuracy reached ~79